# 🛒 Phase 6: Association Rule Learning, Recommendation Systems & PCA
### Course: Artificial Intelligence & Machine Learning (24CSE0316) | Continuous Evaluation
**Project: PlacementIQ AI**

---
## 📑 Syllabus Topics Covered:
1. **Introduction to Association Rule Learning & Frequent Itemset Mining:** Mining non-trivial co-occurrences.
2. **Apriori Algorithm:** Downward-closure candidate pruning.
3. **FP-Growth Algorithm:** Frequent pattern mining using compact FP-Trees without candidate generation.
4. **Market Basket Analysis:** Student competency baskets predicting `Placement_Success` with Support, Confidence, Lift.
5. **Recommendation Systems:** Content-based course matching using student profile gap vectors.
6. **Dimensionality Reduction & PCA:** Eigenvectors, Eigenvalues, Explained Variance Ratio (Scree Plot) & 2D Projection.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

sns.set_theme(style="whitegrid", palette="muted")
print("✅ Association & PCA libraries loaded!")


In [ ]:
# 1. Market Basket Analysis on Student Profiles
train = pd.read_csv('dataset/train.csv')

baskets = []
for _, row in train.iloc[:5000].iterrows():
    b = []
    if row['CGPA'] >= 8.0: b.append("High_CGPA")
    if row['Coding_Skills'] >= 7: b.append("Strong_Coding")
    if row['Internships'] >= 1: b.append("Has_Internship")
    if row['Projects'] >= 3: b.append("Rich_Projects")
    if row['Certifications'] >= 2: b.append("Multi_Certified")
    if row['Backlogs'] == 0: b.append("Zero_Backlogs")
    b.append("Placement_Success" if row['Placement_Status'] == 'Placed' else "Not_Placed")
    baskets.append(b)

te = TransactionEncoder()
te_ary = te.fit(baskets).transform(baskets)
df_basket = pd.DataFrame(te_ary, columns=te.columns_)

frequent_items = fpgrowth(df_basket, min_support=0.15, use_colnames=True)
rules = association_rules(frequent_items, metric="confidence", min_threshold=0.5)
rules_top = rules[rules['consequents'].apply(lambda x: 'Placement_Success' in x)].sort_values('lift', ascending=False)

print(f"Mined {len(frequent_items)} Frequent Itemsets and {len(rules)} Rules.")
print("\nTop Rules Predicting Placement Success:")
display(rules_top[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(5))


In [ ]:
# 2. Recommendation Systems: Skill-Based Curriculum
courses = pd.DataFrame([
    {"ID": "C1", "Title": "DSA & Competitive Coding", "Category": "Coding"},
    {"ID": "C2", "Title": "Full-Stack Web Development", "Category": "Projects"},
    {"ID": "C3", "Title": "Cloud Computing & DevOps", "Category": "Certifications"},
    {"ID": "C4", "Title": "Business Communication & GD", "Category": "Soft Skills"},
])

def recommend_for_student(student):
    needed = []
    if student['Coding_Skills'] < 6: needed.append("Coding")
    if student['Projects'] < 3: needed.append("Projects")
    if student['Certifications'] < 2: needed.append("Certifications")
    if student['Soft_Skills_Rating'] < 6: needed.append("Soft Skills")
    return courses[courses['Category'].isin(needed)]

sample = {'Coding_Skills': 4, 'Projects': 1, 'Certifications': 0, 'Soft_Skills_Rating': 8}
print("Recommended Curriculum for Student Profile:")
display(recommend_for_student(sample))


In [ ]:
# 3. PCA Dimensionality Reduction
features = ['CGPA', 'Coding_Skills', 'Internships', 'Projects',
            'Communication_Skills', 'Aptitude_Test_Score', 'Soft_Skills_Rating',
            'Certifications', 'Backlogs']
X_scaled = StandardScaler().fit_transform(train[features])
y_labels = (train['Placement_Status'] == 'Placed').astype(int)

pca = PCA()
pca.fit(X_scaled)
var_ratio = pca.explained_variance_ratio_

pca_2d = PCA(n_components=2)
X_2d = pca_2d.fit_transform(X_scaled)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Scree Plot
comps = range(1, len(var_ratio) + 1)
ax1.bar(comps, var_ratio, alpha=0.7, color='steelblue', label='Individual')
ax1.plot(comps, np.cumsum(var_ratio), 'ro-', linewidth=2, label='Cumulative')
ax1.axhline(0.80, color='green', linestyle='--', label='80% Threshold')
ax1.set_title("PCA Scree Plot & Cumulative Explained Variance", fontweight='bold')
ax1.set_xlabel("Principal Component")
ax1.set_ylabel("Variance Ratio")
ax1.legend()

# 2D Projection
sub = np.random.choice(len(X_2d), 1500, replace=False)
sc = ax2.scatter(X_2d[sub, 0], X_2d[sub, 1], c=y_labels[sub], cmap='coolwarm', alpha=0.6)
ax2.set_title("2D Projection onto Top 2 Principal Components", fontweight='bold')
ax2.set_xlabel("PC1")
ax2.set_ylabel("PC2")
plt.colorbar(sc, ax=ax2, label="Placement Status")

plt.tight_layout()
plt.show()
